In [2]:
import pandas as pd
import os

DATA_DIR    = r"C:\Users\semwi\FPL-Core-Insights\data"
MATCH_PATH  = os.path.join(DATA_DIR, "match_data.csv")
OUTPUT_PATH = os.path.join(DATA_DIR, "team_rolling_stats.csv")

df = pd.read_csv(MATCH_PATH)

# Alleen gespeelde wedstrijden meenemen
df = df[df["status"].isin(["Ended", "finished"])].copy().reset_index(drop=True)
print(f"📂 {len(df)} gespeelde wedstrijden geladen")

exclude = ["match_id", "season", "round", "timestamp", "status", "home_team", "away_team",
           "home_team_id", "away_team_id", "venue", "referee", "attendance",
           "home_goals", "away_goals",
           "home_elo_pre", "away_elo_pre", "elo_diff_pre",
           "home_elo_post", "away_elo_post", "elo_diff_post"]

stat_cols_home = [c for c in df.columns if c.startswith("home_") and c not in exclude]
stat_cols_away = [c for c in df.columns if c.startswith("away_") and c not in exclude]
stat_names = [c.replace("home_", "") for c in stat_cols_home]

# Drempel: >80% gevuld
threshold = 0.8 * len(df)
stat_names = [s for s in stat_names
              if df[f"home_{s}"].notna().sum() >= threshold
              and df[f"away_{s}"].notna().sum() >= threshold]

def make_long(df, side):
    is_home = side == "home"
    d = df.copy()
    d["team"]         = d["home_team"] if is_home else d["away_team"]
    d["opponent"]     = d["away_team"] if is_home else d["home_team"]
    d["goals_for"]    = d["home_goals"] if is_home else d["away_goals"]
    d["goals_against"]= d["away_goals"] if is_home else d["home_goals"]
    d["win"]  = (d["goals_for"] > d["goals_against"]).astype(int)
    d["draw"] = (d["goals_for"] == d["goals_against"]).astype(int)
    d["loss"] = (d["goals_for"] < d["goals_against"]).astype(int)
    d["side"] = side
    prefix = "home_" if is_home else "away_"
    rename = {f"{prefix}{s}": s for s in stat_names}
    d = d.rename(columns=rename)
    base = ["match_id", "season", "round", "timestamp", "team", "opponent", "side",
            "goals_for", "goals_against", "win", "draw", "loss"]
    return d[base + stat_names]

home_long = make_long(df, "home")
away_long = make_long(df, "away")
overall   = pd.concat([home_long, away_long], ignore_index=True)

roll_cols = ["goals_for", "goals_against", "win", "draw", "loss"] + stat_names

def add_rolling(group, suffix=""):
    group = group.sort_values(["timestamp", "match_id"]).reset_index(drop=True)
    for col in roll_cols:
        group[f"{col}_alltime{suffix}"] = group[col].expanding().mean().shift(1).round(3)
        group[f"{col}_last10{suffix}"]  = group[col].rolling(10, min_periods=1).mean().shift(1).round(3)
    return group

# Overall (thuis + uit gecombineerd)
overall_rolled = []
for team, group in overall.groupby("team"):
    overall_rolled.append(add_rolling(group, suffix="_overall"))
overall_rolled = pd.concat(overall_rolled, ignore_index=True)

# Thuis
home_rolled = []
for team, group in home_long.groupby("team"):
    home_rolled.append(add_rolling(group, suffix="_home"))
home_rolled = pd.concat(home_rolled, ignore_index=True)

# Uit
away_rolled = []
for team, group in away_long.groupby("team"):
    away_rolled.append(add_rolling(group, suffix="_away"))
away_rolled = pd.concat(away_rolled, ignore_index=True)

# Combineer alles
base_cols = ["match_id", "season", "round", "timestamp", "team", "opponent",
             "side", "goals_for", "goals_against", "win", "draw", "loss"]

overall_cols = [c for c in overall_rolled.columns if "_overall" in c]
home_cols    = [c for c in home_rolled.columns if "_home" in c]
away_cols    = [c for c in away_rolled.columns if "_away" in c]

overall_rolled["_key"] = overall_rolled["match_id"].astype(str) + "_" + overall_rolled["team"]
home_rolled["_key"]    = home_rolled["match_id"].astype(str) + "_" + home_rolled["team"]
away_rolled["_key"]    = away_rolled["match_id"].astype(str) + "_" + away_rolled["team"]

base = overall_rolled[base_cols + ["_key"] + overall_cols]
base = base.merge(home_rolled[["_key"] + home_cols], on="_key", how="left")
base = base.merge(away_rolled[["_key"] + away_cols], on="_key", how="left")
base = base.drop(columns=["_key"])

final = base.sort_values(["timestamp", "match_id", "side"]).reset_index(drop=True)
final.to_csv(OUTPUT_PATH, index=False)
print(f"✅ team_rolling_stats.csv: {len(final)} rijen, {len(final.columns)} kolommen")
print(f"   Teams: {final['team'].nunique()}")
print(f"   Laatste wedstrijd: {final['timestamp'].max()}")
print(f"   Kolom voorbeeld: {[c for c in final.columns if 'goals_for' in c]}")

📂 2580 gespeelde wedstrijden geladen
✅ team_rolling_stats.csv: 5160 rijen, 246 kolommen
   Teams: 28
   Laatste wedstrijd: 2026-03-15 16:30
   Kolom voorbeeld: ['goals_for', 'goals_for_alltime_overall', 'goals_for_last10_overall', 'goals_for_alltime_home', 'goals_for_last10_home', 'goals_for_alltime_away', 'goals_for_last10_away']
